In [1]:
import os 
import sys 
import glob 
import re
import nltk 
import logging 

import numpy as np 
import pandas as pd 

os.chdir(r'C:\Users\ADMIN\Documents\Nam_3\CS419\project')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from collections import Counter
from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex


In [2]:
docs_fp = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\documents.csv'
queries_fp = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\queries.csv'

docs_df = pd.read_csv(docs_fp)
queries_df = pd.read_csv(queries_fp)  

In [4]:
docs_processing = TextPreprocessing() 
processed_docs = docs_processing.process_dataframe(dataframe=docs_df) 
vocab = docs_processing.get_vocab() 
processed_docs

,id,content,tokens
0,1,experimental investigation of the aerodynamics...,"[experimental, investigation, aerodynamics, wi..."
1,10,the theory of the impact tube at low pressure ...,"[theory, impact, tube, low, pressure, theoreti..."
2,100,vibration isolation of aircraft power plants ....,"[vibration, isolation, aircraft, power, plant,..."
3,1000,free flight measurements of the static and dyn...,"[free, flight, measurement, static, dynamic, s..."
4,1001,wind tunnel investigation of the static and dy...,"[wind, tunnel, investigation, static, dynamic,..."
...,...,...,...
1395,995,,[]
1396,996,extension of boundary layer separation criteri...,"[extension, boundary, layer, separation, crite..."
1397,997,experimental and theoretical studies of axisym...,"[experimental, theoretical, study, axisymmetri..."
1398,998,"equations, tables and charts for compressible ...","[equation, table, chart, compressible, flow, r..."


In [5]:
def _build_term_docs(vocab, docs_df): 
    term_docs = [] 
    docs_size = docs_df.shape[0]
    for i in range(len(vocab)): 
        r = [0] * docs_size 
        term_docs.append(r) 

    for i in range(docs_size): 
        for w in docs_df['tokens'].iloc[i]: 
            for j in range(len(vocab)): 
                if vocab[j] == w : 
                    term_docs[j][i] += 1
                    break
                
    return term_docs

term_docs = _build_term_docs(vocab=vocab, docs_df=processed_docs) 


In [6]:
type(term_docs)

list

In [7]:
sample = queries_df.iloc[0]
sample

id                                                         1
content     what similarity laws must be obeyed when cons...
Name: 0, dtype: object

In [8]:
sample_tokens = docs_processing._text_preprocessing(sample['content']) 
sample_tokens

['similarity',
 'law',
 'must',
 'obeyed',
 'constructing',
 'aeroelastic',
 'model',
 'heated',
 'high',
 'speed',
 'aircraft']

In [9]:
def boolean_query(query:str, term_docs_dict:list, vocab:list, operator='OR') -> list:
    query_tokens = docs_processing._text_preprocessing(query)  
    print(f"="*10)
    print(f"Query tokens: ")
    print(query_tokens)
    print(f"="*10)

    word2idx = {word: i for i, word in enumerate(vocab)}
    print(f"="*10)
    print(f"word2idx: ")
    print(word2idx)
    print(f"="*10)

    target_indices = [word2idx[t] for t in query_tokens if t in word2idx]
    print(f"="*10)
    print(f"target_indices : ")
    print(target_indices)
    print(f"="*10)

    if not target_indices:
        return []
    
    matrix = np.array(term_docs_dict)
    res_vector = matrix[target_indices[0]]

    for idx in target_indices[1:]:
        if operator == "AND":
            res_vector = res_vector & matrix[idx]
        else: 
            res_vector = res_vector | matrix[idx]

    matching_doc_indices = np.where(res_vector > 0)[0]
    return matching_doc_indices.tolist()

query = sample['content']
result = boolean_query(query=query, term_docs_dict=term_docs, vocab=vocab, operator="OR") 


Query tokens: 
['similarity', 'law', 'must', 'obeyed', 'constructing', 'aeroelastic', 'model', 'heated', 'high', 'speed', 'aircraft']
word2idx: 
{'ab': 0, 'abbreviated': 1, 'ability': 2, 'ablated': 3, 'ablating': 4, 'ablation': 5, 'ablative': 6, 'able': 7, 'abrupt': 8, 'abruptly': 9, 'absence': 10, 'absent': 11, 'absolute': 12, 'absorbed': 13, 'absorbing': 14, 'absorption': 15, 'abstract': 16, 'abundantly': 17, 'academic': 18, 'accelerated': 19, 'accelerates': 20, 'accelerating': 21, 'acceleration': 22, 'accelerator': 23, 'accelerometer': 24, 'accentuated': 25, 'acceptability': 26, 'acceptable': 27, 'acceptably': 28, 'acceptance': 29, 'accepted': 30, 'accessible': 31, 'accidental': 32, 'accommodate': 33, 'accommodated': 34, 'accommodation': 35, 'accompanied': 36, 'accompanies': 37, 'accompany': 38, 'accompanying': 39, 'accomplish': 40, 'accomplished': 41, 'accord': 42, 'accordance': 43, 'according': 44, 'accordingly': 45, 'account': 46, 'accountable': 47, 'accounted': 48, 'accounting':

In [10]:
result

[2,
 3,
 4,
 5,
 6,
 10,
 13,
 15,
 16,
 19,
 24,
 25,
 31,
 35,
 40,
 43,
 46,
 47,
 49,
 50,
 59,
 64,
 70,
 71,
 72,
 73,
 75,
 81,
 82,
 84,
 87,
 90,
 92,
 95,
 98,
 99,
 100,
 102,
 103,
 105,
 107,
 110,
 113,
 115,
 118,
 125,
 126,
 128,
 129,
 130,
 137,
 140,
 141,
 151,
 157,
 158,
 161,
 162,
 165,
 170,
 172,
 173,
 174,
 176,
 177,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 191,
 199,
 202,
 203,
 206,
 208,
 214,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 226,
 227,
 228,
 230,
 235,
 237,
 238,
 239,
 242,
 244,
 245,
 246,
 251,
 252,
 253,
 256,
 257,
 258,
 264,
 267,
 268,
 273,
 275,
 277,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 291,
 292,
 294,
 296,
 299,
 300,
 302,
 303,
 306,
 309,
 311,
 313,
 315,
 318,
 319,
 322,
 323,
 324,
 326,
 328,
 329,
 330,
 331,
 332,
 333,
 334,
 336,
 339,
 340,
 341,
 345,
 347,
 348,
 349,
 350,
 351,
 352,
 356,
 358,
 359,
 360,
 362,
 363,
 366,
 371,
 373,
 374,
 375,
 376,
 377,
 378,
 380,
 38

In [11]:
processed_docs.iloc[result]

,id,content,tokens
2,100,vibration isolation of aircraft power plants ....,"[vibration, isolation, aircraft, power, plant,..."
3,1000,free flight measurements of the static and dyn...,"[free, flight, measurement, static, dynamic, s..."
4,1001,wind tunnel investigation of the static and dy...,"[wind, tunnel, investigation, static, dynamic,..."
5,1002,preliminary investigations of spiked bodies at...,"[preliminary, investigation, spiked, body, hyp..."
6,1003,free flight measurements of the static and dyn...,"[free, flight, measurement, static, dynamic, r..."
...,...,...,...
1390,990,a rapid approximate method for determining vel...,"[rapid, approximate, method, determining, velo..."
1391,991,wing flow study of pressure drag reduction at ...,"[wing, flow, study, pressure, drag, reduction,..."
1394,994,investigation of a retrocket exhausting from t...,"[investigation, retrocket, exhausting, nose, b..."
1398,998,"equations, tables and charts for compressible ...","[equation, table, chart, compressible, flow, r..."


## **Inverted Index**

In [14]:
inverted_indx_builder = InvertedIndex(processed_docs_df=processed_docs, vocab=vocab) 
inverted_indx_builder._build() 

inverted_index = inverted_indx_builder.get_inverted_index()

In [26]:
inverted_index['rapid']['df']

32

In [21]:
exp_list = []

In [23]:
a = list(inverted_index['comparative']['postings'].keys())

In [24]:
b = list(inverted_index['retrocket']['postings'].keys())

In [25]:
exp_list.append(np.logical_or(a, b)) 
exp_list

[array([ True,  True,  True,  True,  True,  True])]